In [16]:
import pandas as pd
import sqlite3
from nba_api.stats.endpoints import teamgamelog

In [17]:
from nba_api.stats.endpoints import leaguegamelog

def get_team_game_stats_fast(season):
    """
    Pulls all team-level game logs for a given season.
    Includes ORtg, DRtg, Pace, and Four Factors.
    """
    tgl = leaguegamelog.LeagueGameLog(season=season)
    df = tgl.get_data_frames()[0]

    df["SEASON"] = season

    return df

In [18]:
def add_header_row(df, season):
    header = {col: None for col in df.columns}
    header["TEAM_ABBREVIATION"] = "SEASON"
    header["SEASON"] = season
    df = pd.concat([pd.DataFrame([header]), df], ignore_index=True)
    return df

In [19]:
def save_team_game_stats(df, season):
    table_name = f"team_game_stats_{season.replace('-', '_')}"

    conn = sqlite3.connect("nba.db")
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()

    df.to_csv(f"../data/raw/team_game_stats_{season}.csv", index=False)

    print(f"Saved: {table_name} and team_game_stats_{season}.csv")

In [20]:
season = [
    "2009-10",
]

for s in season:
    df = get_team_game_stats_fast(s)
    df = add_header_row(df, s)
    save_team_game_stats(df, s)

    df.head()

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)